# Домашнее задание № 2. Что расскажут данные о вакансиях?

Вы анализируете учебную выгрузку вакансий. Нужно проверить качество данных, найти интересные различия между вакансиями и подтвердить выводы расчётами.

**Результат:** тетрадка `.ipynb` с выполненными ячейками, таблицами, одним графиком и короткими ответами. Задания 1–10 обязательны. Дополнительные задания в конце не входят в оценку 6–8 часов.

## Порядок выполнения
1. Положите **исходный `vacancies.csv`** рядом с ноутбуком. В Colab загрузите его через панель «Файлы». В этот ноутбук CSV не встроен.
2. Выполняйте ячейки сверху вниз. Прочитайте пример на маленькой таблице, затем примените приём к вакансиям. Примеры не меняют основной DataFrame.
3. После каждого задания запишите ответ и 1–2 предложения вывода. Код нужен для воспроизводимости, вывод — чтобы объяснить результат.
4. Перед сдачей перезапустите ядро и выполните все ячейки. Сохраните файл как `Фамилия_Группа_Pandas.ipynb` с результатами выполнения.

In [ ]:
import ast
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_PATH = Path('vacancies.csv')
if not DATA_PATH.is_file():
    raise FileNotFoundError('Загрузите исходный vacancies.csv в папку с ноутбуком и запустите ячейку снова.')

## Что находится в данных
| Столбец | Содержание |
|---|---|
| `position_name` | Название вакансии |
| `employer_name` | Компания |
| `area` | Город или регион |
| `experience` | Требуемый опыт |
| `schedule` | График работы |
| `employment` | Тип занятости |
| `professional_roles` | Категория профессиональной роли |
| `salary` | Зарплатное предложение: строка со словарём или пропуск |
| `key_skills` | Навыки: строка со списком |
| `description` | Описание вакансии |

Служебный `Unnamed: 0`, если он есть, — сохранённый индекс; перед проверкой дубликатов его нужно удалить. Одна строка означает одну запись вакансии в выгрузке. Разные вакансии могут иметь одинаковое название.

## 1. Загрузите и изучите данные · 6 баллов
1. Загрузите CSV в переменную `data` с помощью `pd.read_csv`. Для исходного файла используется запятая как разделитель.
2. Покажите первые 5 строк, выполните `info()` и `describe()`.
3. Определите число строк и столбцов **до удаления служебного индекса**.
4. Выведите названия столбцов и три значения `salary` и `key_skills`. Объясните, почему эти поля пока нельзя обрабатывать как обычные числа или списки.

In [ ]:
demo = pd.DataFrame({'sensor':['A','B','C'], 'temperature':[21.5, 23.0, None]})
display(demo.head(2))
print('Строк и столбцов:', demo.shape)
demo.info()
display(demo.describe(include='all'))
print(demo.columns.tolist())

In [ ]:
# Ваше решение здесь

**Ответ и вывод:** 

## 2. Проверьте пропуски и удалите полные дубликаты · 12 баллов
1. Удалите `Unnamed: 0`, если столбец присутствует.
2. Посчитайте пропуски в каждом столбце. Назовите столбец с наибольшим числом пропусков и долю его пропусков в процентах.
3. Посчитайте полные дубликаты после удаления служебного индекса.
4. Создайте `data_cleaned`: удалите только полные дубликаты и сделайте `.copy()`. Не удаляйте записи только из-за отсутствия зарплаты.
5. Укажите число строк до и после очистки. Проверьте: исходное число строк минус число удалённых дублей равно числу оставшихся строк.

**Важно:** всё дальнейшее задание выполняйте на `data_cleaned`. Преобразование навыков в списки запускается после удаления дублей, потому что списки неудобны для этой проверки.

In [ ]:
demo_quality = pd.DataFrame({'device':['A','B','A'], 'value':[10, None, 10]})
display(demo_quality.isna().sum())
print('Доля пропусков value, %:', demo_quality['value'].isna().mean() * 100)
print('Дубликаты:', demo_quality.duplicated().sum())
demo_cleaned = demo_quality.drop_duplicates().copy()
display(demo_cleaned)
# Если столбец не обязателен: df = df.drop(columns=['служебный_столбец'], errors='ignore')

In [ ]:
# Ваше решение здесь

**Ответ и вывод:** 

## Готовая подготовка вложенных полей — просто запустите
Этот блок преобразует строки со списками и словарями. **Разрабатывать эти функции и разбирать их код для сдачи не требуется.** Не используйте `eval()`.

После запуска появятся:
- `key_skills_list` — настоящий список навыков; пустой список означает, что навыки не перечислены, а не что они не нужны;
- `salary_from`, `salary_to` — числовые границы предложения, отсутствующая граница остаётся пропуском;
- `currency` — исходный код валюты.

В этой выгрузке рублёвые предложения имеют код **`RUR`**. Валюты не конвертируем. Исходные `salary` и `key_skills` сохраняются.

In [ ]:
def read_nested(value, expected_type, empty_value, field, row_index):
    if isinstance(value, expected_type):
        return value
    if value is None or (not isinstance(value, (list, dict)) and pd.isna(value)):
        return empty_value.copy()
    if isinstance(value, str) and value.strip().lower() in ('', 'null', 'none', 'nan'):
        return empty_value.copy()
    for parser in (ast.literal_eval, json.loads):
        try:
            result = parser(value)
        except (ValueError, SyntaxError, TypeError):
            continue
        if isinstance(result, expected_type):
            return result
    raise ValueError(f'Не удалось разобрать {field}, индекс строки {row_index}: {str(value)[:100]}')

def prepare_nested_fields(df):
    result = df.copy()
    salary_dicts = [read_nested(v, dict, {}, 'salary', i) for i, v in result['salary'].items()]
    skills_lists = [read_nested(v, list, [], 'key_skills', i) for i, v in result['key_skills'].items()]
    for i, skills in zip(result.index, skills_lists):
        if not all(isinstance(skill, str) for skill in skills):
            raise ValueError(f'В key_skills ожидаются строки: индекс {i}')
    result['key_skills_list'] = skills_lists
    for key in ('from', 'to'):
        result['salary_' + key] = pd.to_numeric(
            pd.Series([s.get(key) for s in salary_dicts], index=result.index), errors='coerce'
        )
    result['currency'] = [s.get('currency') for s in salary_dicts]
    return result

data_cleaned = prepare_nested_fields(data_cleaned)
display(data_cleaned[['salary', 'salary_from', 'salary_to', 'currency', 'key_skills_list']].head())

## 3. Посчитайте категории и частоты · 8 баллов
1. Найдите число уникальных значений `professional_roles`, не считая пропуски.
2. Постройте таблицу количества вакансий по `experience`.
3. Выведите 5 городов или регионов с наибольшим числом вакансий.
4. Какой уровень опыта встречается чаще всего? Подтвердите ответ количеством вакансий.

Подсказка: `nunique()` считает разные значения, а `value_counts()` — частоту каждого значения. Для показа пропусков в частотной таблице используйте `dropna=False`.

In [ ]:
demo_categories = pd.Series(['Москва', 'Казань', 'Москва', 'Тула', None])
print('Разных непустых городов:', demo_categories.nunique())
display(demo_categories.value_counts(dropna=False))
display(demo_categories.value_counts().head(2))

In [ ]:
# Ваше решение здесь

**Ответ и вывод:** 

## 4. Отберите вакансии по условиям · 12 баллов
1. Посчитайте вакансии с `experience == 'Нет опыта'` и их долю среди всех очищенных записей.
2. Отберите вакансии одновременно без опыта и с `schedule == 'Удаленная работа'`.
3. Выведите число таких записей и первые 5 строк, оставив `position_name`, `employer_name`, `area`, `experience`, `schedule`.
4. Найдите их долю **среди вакансий без опыта**. Объясните, почему этот процент отличается от доли среди всех вакансий.

Если подходящих записей нет, это тоже ответ. Если знаменатель равен нулю, напишите, что долю вычислить нельзя. Для объединения условий используйте `&`, а каждое условие заключайте в скобки.

In [ ]:
demo_filter = pd.DataFrame({'device':['A','B','C'], 'value':[5,12,18], 'room':['X','Y','X']})
mask = (demo_filter['value'] > 10) & (demo_filter['room'] == 'X')
display(demo_filter.loc[mask, ['device', 'value']])
print('Подходящих строк:', mask.sum())
print('Доля среди всех строк, %:', mask.mean() * 100)

In [ ]:
# Ваше решение здесь

**Ответ и вывод:** 

## 5. Создайте признаки на основе навыков · 12 баллов
1. Создайте `skills_count`: количество элементов в `key_skills_list`.
2. Найдите среднее число перечисленных навыков и число вакансий с пустым списком.
3. Создайте `is_python_requirement`: `True`, если один из навыков после удаления пробелов по краям и приведения регистра равен `python`; иначе `False`. Здесь ищем точное название навыка, а не подстроку.
4. Найдите количество и процент таких вакансий среди всех очищенных записей.
5. Сравните среднее число перечисленных навыков в вакансиях с Python и без него.

Проверка: записи со списками `['Python', 'SQL']` и `[' PYTHON ']` должны получить `True`, с `['SQL']` и `[]` — `False`.

In [ ]:
demo_lists = pd.Series([['Wi-Fi','Bluetooth'], ['USB'], []])
display(demo_lists.apply(len))

def has_usb(items):
    for item in items:
        if item.strip().lower() == 'usb':
            return True
    return False

flags = demo_lists.apply(has_usb)
display(flags)
print('Доля списков с USB, %:', flags.mean() * 100)

In [ ]:
# Ваше решение здесь

**Ответ и вывод:** 

## 6. Рассчитайте показатель зарплатного предложения · 14 баллов
1. Выведите используемые коды валют и число вакансий с каждым кодом; пропуски покажите отдельной строкой.
2. Создайте `salary_num` по правилам:
   - обе границы указаны → их среднее;
   - только одна граница указана → её значение;
   - обе отсутствуют → пропуск, **не ноль**.
3. Создайте `salary_rur`: значение `salary_num` только для `currency == 'RUR'`, в остальных строках — пропуск.
4. Найдите число записей с непустым `salary_rur`, их среднее и медиану. Не усредняйте разные валюты вместе.
5. Выведите первые 5 вакансий с наибольшим `salary_rur`, показав название, компанию и обе исходные границы.
6. Напишите, какая оценка (среднее или медиана) лучше описывает типичное предложение в вашей выборке, и обоснуйте ответ полученными числами.

In [ ]:
demo_bounds = pd.DataFrame({'left':[10,20,None,None], 'right':[30,None,40,None]})
# Среднее по строке: axis=1. Пропущенная граница не участвует в среднем.
demo_bounds['midpoint'] = demo_bounds[['left','right']].mean(axis=1)
display(demo_bounds)

demo_prices = pd.DataFrame({'amount':[100,200,300], 'unit':['A','B','A']})
demo_prices['amount_a'] = demo_prices['amount'].where(demo_prices['unit'] == 'A')
display(demo_prices)
print('Среднее:', demo_prices['amount_a'].mean())
print('Медиана:', demo_prices['amount_a'].median())
display(demo_prices.sort_values('amount', ascending=False).head(2))

In [ ]:
# Ваше решение здесь

**Ответ и вывод:** 

## 7. Сравните компании · 10 баллов
1. Выведите 5 компаний с наибольшим числом вакансий **во всей очищенной таблице**.
2. Отдельно выведите 5 компаний с наибольшим числом вакансий с непустым `salary_rur`.
3. Сравните лидеров двух списков. Объясните, почему они могут различаться.
4. Для одной компании из первого списка найдите количество всех её вакансий и долю с непустым `salary_rur`.

Не подменяйте число вакансий числом уникальных названий: две разные записи могут иметь одно название. Не называйте отсутствие рублёвой зарплаты отсутствием любой зарплаты. При равенстве максимальных количеств укажите всех лидеров.

In [ ]:
demo_sales = pd.DataFrame({'shop':['A','A','B','C'], 'price':[100,None,200,300]})
display(demo_sales['shop'].value_counts())
with_price = demo_sales.loc[demo_sales['price'].notna()]
display(with_price['shop'].value_counts())

In [ ]:
# Ваше решение здесь

**Ответ и вывод:** 

## 8. Сгруппируйте вакансии по опыту · 12 баллов
Постройте одну таблицу по `experience` со столбцами:
1. `vacancies_count` — количество **всех** записей в группе;
2. `salary_rur_count` — число записей с непустым `salary_rur`;
3. `avg_salary_rur` — средний `salary_rur`;
4. `companies_count` — число уникальных непустых названий компаний.

Группируйте всю очищенную таблицу. Не удаляйте заранее строки без зарплаты: они нужны для подсчёта вакансий и компаний. `mean` и `count` сами пропускают отсутствующие значения. Используйте `dropna=False`, чтобы строки без уровня опыта тоже вошли в таблицу.

Проверьте: сумма `vacancies_count` равна числу строк `data_cleaned`. Сделайте один вывод и укажите размер зарплатной подвыборки, на которую он опирается. Группа без зарплат получает пропуск среднего, а не ноль.

In [ ]:
demo_groups = pd.DataFrame({'kind':['A','A','B','B'], 'price':[100,None,200,400], 'shop':['X','Y','X','X']})
demo_summary = demo_groups.groupby('kind', dropna=False).agg(
    rows_count=('kind','size'),
    price_count=('price','count'),
    avg_price=('price','mean'),
    shops_count=('shop','nunique')
)
display(demo_summary)
print('Проверка суммы строк:', demo_summary['rows_count'].sum() == len(demo_groups))

In [ ]:
# Ваше решение здесь

**Ответ и вывод:** 

## 9. Постройте график · 8 баллов
Постройте горизонтальную столбчатую диаграмму для 5 компаний с наибольшим числом вакансий во всей очищенной таблице. Если компаний меньше пяти, покажите все.
- Подпишите оси и добавьте содержательный заголовок.
- Убедитесь, что названия компаний читаются и шкала количества начинается с нуля.
- Напишите 2 предложения: что показывает график и какой вывод он позволяет сделать.

Числа на графике должны совпасть с таблицей задания 7.

In [ ]:
demo_counts = pd.Series([12,8,5], index=['Магазин A','Магазин B','Магазин C'])
ax = demo_counts.sort_values().plot.barh(figsize=(8,4), color='#3B6EA5')
ax.set_title('Количество товаров по магазинам — учебный пример')
ax.set_xlabel('Количество товаров')
ax.set_ylabel('Магазин')
ax.set_xlim(left=0)
plt.tight_layout()
plt.show()

In [ ]:
# Ваше решение здесь

**Ответ и вывод:** 

## 10. Подведите итоги и проверьте ноутбук · 6 баллов
Напишите краткие выводы (150–250 слов):
1. Какие три наиболее интересных факта вы нашли? Каждый подтвердите числом из своих расчётов.
2. Какие две особенности данных ограничивают выводы? Например, пропуски зарплаты, неполные списки навыков или небольшой объём зарплатной подвыборки.

Отделяйте наблюдения от предположений. Не делайте причинных выводов из простого сравнения средних.

Перед сдачей:
- замените все поля «Ответ и вывод» своими ответами;
- удалите или очистите ячейки с ошибками, исправив их причины;
- перезапустите ядро, выполните всё сверху вниз и сохраните результаты;
- проверьте, что решения используют `data_cleaned` и не смешивают валюты;
- сдайте `Фамилия_Группа_Pandas.ipynb`. Отдельный отчёт и скриншоты не нужны.

**Выводы:**

Ваш текст здесь:

## Дополнительно — по желанию, вне обязательных 6–8 часов
**А. Топ навыков.** Найдите 10 самых часто перечисленных навыков. Сначала приведите названия к единому регистру и уберите пробелы. Один навык учитывайте не более одного раза в вакансии. Подсказка: `apply`, затем `explode()` и `value_counts()`. Пустые списки не должны стать отдельным навыком.

**Б. Сводная таблица.** С помощью `pd.pivot_table` сравните медианный `salary_rur` по уровню опыта и графику работы. Рядом покажите число непустых рублёвых предложений в каждой ячейке. Не заменяйте отсутствующие зарплатные оценки нулями.

Эти задания — для дополнительной практики; их отсутствие не снижает максимальную оценку основной работы.

## Критерии оценивания — 100 баллов
| Задание | Баллы | Что проверяется |
|---|---:|---|
| 1. Обзор | 6 | Загрузка, структура и понимание формата полей |
| 2. Качество | 12 | Пропуски, корректное удаление дублей, сохранение строк без зарплаты |
| 3. Категории | 8 | Уникальные значения и частоты |
| 4. Фильтры | 12 | Два условия и правильные знаменатели процентов |
| 5. Признаки | 12 | Число навыков, точный поиск Python, проценты и сравнение |
| 6. Зарплата | 14 | Обе/одна/ни одной границы, валюта, среднее, медиана и объяснение |
| 7. Компании | 10 | Подсчёт записей, два списка и доля для выбранной компании |
| 8. Группировка | 12 | Различие size/count/nunique, среднее только в рублях, проверка суммы |
| 9. График | 8 | Верные числа, читаемые подписи и вывод |
| 10. Итоги | 6 | Три факта, две оговорки и воспроизводимость ноутбука |

Разрешено пользоваться документацией и примерами этого ноутбука. Вы должны понимать своё решение и уметь объяснить, откуда берутся результаты.